# 01. EDA — 파인블랭킹 프레스 유압펌프 이상탐지

## 문제
유압펌프 이상 → 프레스 압력 불안정 → 불량·비가동. 현재는 주기점검/작업자 경험에 의존하므로
**초기 이상징후를 놓치거나(미탐지), 정상적인 운전변화를 이상으로 오인(오경보)** 할 위험이 있다.
모터 상·하부 진동(AI0/AI1)과 전류(AI2)로 이상을 조기 탐지하되, **오경보를 줄이고
미탐지·오경보의 발생조건을 설명**하는 것이 최종 목표다.

## 이 노트북에서 답할 것
1. **로드** — 두 파일을 읽고 형태를 확인한다.
2. **기본 점검** — 0.1초 샘플링이 맞는지, 결측·중복·정렬이 어떤지 *직접 확인*한다.
3. **핵심 질문** — normal과 outlier는 정말 무관한 데이터인가?
   - 가설 A: 두 파일의 timestamp가 겹치는 구간이 있다.
   - 가설 B: outlier 파일 중간에 비어 있는 부분은 *원래 normal 구간이었는데 삭제된 것*이다.
   - 두 가설을 검증 가능한 형태로 바꿔 데이터로 판정한다.
4. **추가로 봐야 할 것** — 모델링 전에 확인해야 할 항목을 근거와 함께 제안한다.

> 결론을 먼저 말하면, 3번의 두 가설은 모두 **기각**된다. 다만 기각되는 이유 자체가
> 이 과제에서 가장 중요한 제약 조건이 되므로, 섹션 4~5에서 근거를 하나씩 쌓아간다.

## 0. 설정

의도: 경로·상수를 한곳에 모아 두고, 이후 노트북(`02_`, `03_`…)에서도 같은 정의를 재사용한다.
`GAP_THR`(취득 중단으로 볼 간격)은 섹션 3에서 데이터로 근거를 확인한 뒤 고정한다.

In [ ]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

# 한글 폰트 (WSL: fonts-nanum)
_avail = {f.name for f in font_manager.fontManager.ttflist}
for _f in ["NanumGothic", "NanumBarunGothic", "Malgun Gothic", "AppleGothic", "DejaVu Sans"]:
    if _f in _avail:
        mpl.rcParams["font.family"] = _f
        break
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.titlesize": 10, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8})

DATA_DIR = Path(os.environ.get("KAMP_DATA_DIR", "/mnt/d/data/kamp_ai/press_anomaly_dataset"))
SIGNALS  = ["AI0_Vibration", "AI1_Vibration", "AI2_Current"]
FS       = 10.0    # Hz, 0.1초 샘플링 가정 — 섹션 3에서 검증
GAP_THR  = 0.15    # 초. 이보다 큰 간격은 '취득 중단'으로 간주
COLOR    = {"normal": "#1f77b4", "outlier": "#d62728"}

print("폰트      :", mpl.rcParams["font.family"])
print("DATA_DIR  :", DATA_DIR, "| exists:", DATA_DIR.exists())
print("파일      :", sorted(p.name for p in DATA_DIR.glob("*.csv")))

## 1. 데이터 로드

의도: 첫 컬럼은 이름 없는 원본 행번호이므로 `index_col=0`으로 받고, `TimeStamp`는 문자열이 아닌
datetime으로 파싱해 둔다. 이후 모든 시간 연산(간격·겹침·세그먼트)이 이 파싱에 의존한다.

In [ ]:
normal  = pd.read_csv(DATA_DIR / "press_data_normal.csv", index_col=0, parse_dates=["TimeStamp"])
outlier = pd.read_csv(DATA_DIR / "outlier_data.csv",      index_col=0, parse_dates=["TimeStamp"])
DFS = {"normal": normal, "outlier": outlier}

for k, df in DFS.items():
    print(f"[{k:<7}] shape={str(df.shape):<10} {df.TimeStamp.min()}  ~  {df.TimeStamp.max()}")

display(normal.head(3))
display(outlier.head(3))

## 2. 기본 구조 점검

의도: "0.1초 단위이고 null은 없다"는 사전 확인을 **재현 가능한 형태로 다시 검증**한다.
동시에 흔히 놓치는 항목을 같이 본다 — 행 중복, timestamp 중복, 인덱스 연속성, 시간 정렬,
그리고 `Equipment_state`가 행 단위 라벨인지 파일 단위 상수인지.
마지막 항목은 모델링 설계를 통째로 바꾸므로 반드시 먼저 확인해야 한다.

In [ ]:
def basic_profile(df: pd.DataFrame, name: str) -> pd.Series:
    ts = df["TimeStamp"]
    return pd.Series({
        "rows"              : len(df),
        "null 총합"          : int(df.isna().sum().sum()),
        "완전중복 행"        : int(df.duplicated().sum()),
        "중복 timestamp"     : int(ts.duplicated().sum()),
        "index == 0..n-1"   : bool((df.index.values == np.arange(len(df))).all()),
        "시간 오름차순"      : bool(ts.is_monotonic_increasing),
        "관측 구간(초)"      : round((ts.max() - ts.min()).total_seconds(), 1),
        "날짜"              : ", ".join(sorted({str(d) for d in ts.dt.date})),
        "Equipment_state"   : str(ts.index.size and df["Equipment_state"].value_counts().to_dict()),
    }, name=name)

display(pd.concat([basic_profile(df, k) for k, df in DFS.items()], axis=1))

In [ ]:
# 중복 timestamp가 잡혔다면 실체를 눈으로 확인한다 (값까지 같은 진짜 중복인지, 동시각 2채널인지)
for k, df in DFS.items():
    m = df["TimeStamp"].duplicated(keep=False)
    if m.any():
        print(f"[{k}] 중복 timestamp {int(m.sum())}행")
        display(df[m])
    else:
        print(f"[{k}] 중복 timestamp 없음")

### 2-1. 확인 결과

- **null 0, 완전 정렬, 인덱스 연속** — 사전 확인대로다.
- `normal`에 **timestamp가 같고 값까지 동일한 중복 1쌍**(idx 11564/11565)이 있다. 1/20000이라 통계에는
  영향이 없지만, 윈도우를 만들 때 한 칸씩 밀리므로 전처리에서 제거 대상으로 기록해 둔다.
- **`Equipment_state`는 파일 단위 상수다** (normal 전부 0, outlier 전부 600행 모두 1).
  → 이상이 *언제* 시작됐는지에 대한 행 단위 정답이 없다. 지도학습 이진분류가 아니라
  **정상 데이터로 기준을 학습하는 이상탐지(one-class)** 로 접근해야 한다는 뜻이다.

## 3. 샘플링 간격 점검

의도: "0.1초 단위"가 *모든 구간에서* 성립하는지 본다. 시계열에서 결측은 `NaN`으로만 오지 않는다 —
**행 자체가 없는 형태(시간 구멍)** 가 훨씬 흔하고, `isna()`로는 절대 잡히지 않는다.
연속된 timestamp 차이의 분포를 보면 바로 드러난다.

In [ ]:
DT = {k: df["TimeStamp"].diff().dt.total_seconds() for k, df in DFS.items()}

def interval_report(k: str) -> pd.Series:
    d    = DT[k]
    gaps = d[d > GAP_THR]
    span = (DFS[k].TimeStamp.max() - DFS[k].TimeStamp.min()).total_seconds()
    return pd.Series({
        "dt 중앙값(초)"        : d.median(),
        "dt 최빈값(초)"        : d.round(3).mode().iloc[0],
        "dt == 0.1 비율"       : round((d.round(3) == 0.1).mean(), 4),
        "dt 최소/최대(초)"     : f"{d.min():.3f} / {d.max():.3f}",
        f"gap(>{GAP_THR}s) 개수": int(len(gaps)),
        "gap 총합(초)"         : round(gaps.sum(), 1),
        "gap이 차지한 비율"     : round(gaps.sum() / span, 3),
        "gap 중앙값(초)"       : round(gaps.median(), 2) if len(gaps) else np.nan,
    }, name=k)

display(pd.concat([interval_report(k) for k in DFS], axis=1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.0))
for ax, k in zip(axes, DFS):
    ax.plot(np.arange(len(DT[k])), DT[k].values, lw=.6, color=COLOR[k])
    ax.axhline(0.1, color="k", ls="--", lw=.8)
    ax.set_yscale("log")
    ax.set_title(f"{k} — 연속 샘플 간격 dt  (점선 = 0.1s)")
    ax.set_xlabel("행 번호"); ax.set_ylabel("dt [s], log")
plt.tight_layout(); plt.show()

### 3-1. 확인 결과 — 사전 가정을 하나 수정해야 한다

두 파일 모두 **dt의 97% 정도는 정확히 0.1초**가 맞다. 그런데 나머지 몇 %가 수 초짜리 구멍이고,
이게 **normal에도 똑같이 있다**:

| | gap 개수 | gap 총합 | 관측구간에서 gap 비율 |
|---|---|---|---|
| normal | **598개** | **2,676초** | 약 58% |
| outlier | 20개 | 108초 | 약 65% |

즉 **"normal은 중간에 비어 있는 게 없다"는 전제는 성립하지 않는다.** normal도 관측구간 77분 중
실제로 데이터가 있는 시간은 2,000초(33분)뿐이고, 나머지는 전부 취득이 멈춰 있다.
그리고 gap 패턴이 두 파일에서 매우 비슷하다 — 다음 섹션에서 이 구조의 정체를 밝힌다.

## 4. 취득 세그먼트(프레스 사이클) 복원

의도: gap을 "결측"으로 보고 메우려 들기 전에, **gap이 왜 생겼는지**를 먼저 규명한다.
gap이 불규칙한 통신 장애라면 보간 대상이지만, 일정한 주기로 반복된다면 그건 결측이 아니라
**장비의 동작 주기(프레스 1타 = 1사이클)에 맞춰 DAQ가 버스트로 수집한 결과**다.
후자라면 gap은 메우면 안 되고, 오히려 **세그먼트를 분석 단위로 삼아야** 한다.

판별 기준: gap 간격(사이클 주기)이 일정한가? 세그먼트 길이에 상한(버퍼 크기)이 보이는가?

In [ ]:
def make_segments(df: pd.DataFrame, gap_thr: float = GAP_THR) -> pd.DataFrame:
    # gap으로 끊어 seg_id를 부여하고, 세그먼트 내 경과시간(t_in_seg)을 붙인다
    d   = df["TimeStamp"].diff().dt.total_seconds()
    out = df.copy()
    out["seg_id"]   = (d > gap_thr).cumsum().values
    out["t_in_seg"] = out.groupby("seg_id").cumcount() / FS
    return out


def segment_table(sdf: pd.DataFrame) -> pd.DataFrame:
    g = sdf.groupby("seg_id")
    t = pd.DataFrame({"n": g.size(), "t_start": g.TimeStamp.min(), "t_end": g.TimeStamp.max()})
    t["dur_sec"]        = (t.t_end - t.t_start).dt.total_seconds()
    t["gap_before_sec"] = (t.t_start - t.t_end.shift()).dt.total_seconds()
    t["cycle_sec"]      = t.t_start.diff().dt.total_seconds()      # 사이클 주기
    return t

SEG  = {k: make_segments(df) for k, df in DFS.items()}
SEGT = {k: segment_table(v)  for k, v in SEG.items()}

summary = pd.DataFrame({
    k: {
        "세그먼트 수"          : len(t),
        "세그먼트 길이 중앙값"  : t.n.median(),
        "세그먼트 길이 최대"    : t.n.max(),
        "길이==50 비율"        : round((t.n == 50).mean(), 3),
        "사이클 주기 중앙값(초)": round(t.cycle_sec.median(), 3),
        "사이클 주기 IQR(초)"   : round(t.cycle_sec.quantile(.75) - t.cycle_sec.quantile(.25), 3),
        "gap 중앙값(초)"       : round(t.gap_before_sec.median(), 3),
        "취득 duty(%)"        : round(100 * t.n.sum() / FS
                                     / (t.t_end.max() - t.t_start.min()).total_seconds(), 1),
    } for k, t in SEGT.items()})
display(summary)

print("outlier 세그먼트 앞부분:")
display(SEGT["outlier"].head(8))

In [ ]:
# 두 파일의 앞 200초를 같은 축에 올려 취득 패턴을 직접 비교한다
fig, axes = plt.subplots(2, 1, figsize=(12, 3.4))
for ax, k in zip(axes, DFS):
    t  = SEGT[k]
    t0 = t.t_start.min()
    a  = (t.t_start - t0).dt.total_seconds().values
    b  = (t.t_end   - t0).dt.total_seconds().values
    ax.hlines(np.zeros_like(a), a, b, color=COLOR[k], lw=10)
    ax.set_title(f"{k} — 취득 세그먼트 (전체 {len(t)}개, 앞 200초 구간)")
    ax.set_yticks([]); ax.set_xlim(0, 200); ax.set_ylim(-1, 1)
    ax.set_xlabel("파일 시작 이후 경과시간 [s]")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.0))
for k in DFS:
    t = SEGT[k]
    axes[0].hist(t.n, bins=np.arange(0, 56, 2), density=True, alpha=.55, label=k, color=COLOR[k])
    axes[1].hist(t.cycle_sec.dropna(), bins=np.arange(0, 20, .5), density=True, alpha=.55,
                 label=k, color=COLOR[k])
    axes[2].hist(t.gap_before_sec.dropna(), bins=np.arange(0, 12, .5), density=True, alpha=.55,
                 label=k, color=COLOR[k])
axes[0].set_title("세그먼트 길이 [샘플]"); axes[0].axvline(50, color="k", ls="--", lw=.8)
axes[1].set_title("사이클 주기 [s]")
axes[2].set_title("gap 길이 [s]")
for ax in axes: ax.legend(); ax.set_ylabel("density")
plt.tight_layout(); plt.show()

의도: 결론을 내리기 전에 gap 하나를 실제로 눈으로 확인한다. 통계만으로는 "행이 정말 없는 것인지,
혹시 이상한 값으로 채워진 건 아닌지"를 확신할 수 없다. normal 데이터의 아주 초반(seg 0 → seg 1 경계)
하나를 뽑아 원본 행을 그대로 출력하고, 파형도 같이 그려 gap 앞뒤가 어떻게 이어지는지 본다.

In [ ]:
sid  = 1                                             # 확인할 세그먼트 경계 (seg 0 -> seg 1)
prev = SEG["normal"].query("seg_id == @sid - 1").tail(5)
curr = SEG["normal"].query("seg_id == @sid").head(5)
gap  = (curr.TimeStamp.iloc[0] - prev.TimeStamp.iloc[-1]).total_seconds()

print(f"seg {sid-1} 마지막 5행 (index {prev.index[0]}~{prev.index[-1]}):")
display(prev[["TimeStamp", "AI0_Vibration", "AI2_Current"]])
print(f"\n>>> 인덱스가 {prev.index[-1]} 다음 바로 {curr.index[0]}로 이어짐(끼어든 행 없음). "
      f"timestamp만 {gap:.3f}초 건너뜀 <<<\n")
print(f"seg {sid} 처음 5행 (index {curr.index[0]}~{curr.index[-1]}):")
display(curr[["TimeStamp", "AI0_Vibration", "AI2_Current"]])

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.plot(prev.TimeStamp, prev.AI0_Vibration, "o-", color=COLOR["normal"])
ax.plot(curr.TimeStamp, curr.AI0_Vibration, "o-", color=COLOR["normal"])
ax.axvspan(prev.TimeStamp.iloc[-1], curr.TimeStamp.iloc[0], color="red", alpha=.12)
ax.set_title(f"normal seg {sid-1} → seg {sid} 경계 — 빨간 구간 {gap:.2f}s은 행 자체가 없는 gap")
ax.set_xlabel("TimeStamp"); ax.set_ylabel("AI0_Vibration")
plt.tight_layout(); plt.show()

### 4-1. 확인 결과 — gap은 결측이 아니라 **취득 duty cycle**이다

세 가지 증거가 같은 방향을 가리킨다.

1. **세그먼트 길이에 정확한 상한 50이 있다** (두 파일 모두 최대 50, normal은 34%가 정확히 50).
   50샘플 = 5.0초 → DAQ 버퍼/트리거 설정값이다. 자연적인 결측이라면 이런 상한이 생기지 않는다.
2. **사이클 주기가 두 파일에서 거의 동일하다** — normal 약 7.96초, outlier 약 8.13초 (중앙값).
   프레스 1타 주기로 보는 게 타당하다.
3. **취득 duty도 유사하다** (둘 다 35~43% 수준).

**결론:** `outlier` 파일의 빈 구간은 *normal 구간이 삭제된 흔적이 아니다*. normal 파일에도
598개의 동일한 구멍이 있고, 구조(길이 상한·주기·duty)까지 같다. 두 파일은 같은 취득 설정에서
서로 다른 시점에 기록된 것이다. → **가설 B 기각**

**실무적 함의**
- gap을 보간하거나 `resample().ffill()`로 메우면 **존재하지 않는 신호를 만들어내는 것**이다. 하지 않는다.
- 윈도우/특징은 반드시 **세그먼트 내부에서만** 만든다. gap을 가로지르는 윈도우는 서로 다른 프레스 타를
  이어붙인 가짜 파형이 된다.
- 자연스러운 분석 단위는 **1 세그먼트 = 1 프레스 사이클**이다. 이상 판정도 사이클 단위가 맞다.

## 5. 핵심 질문 — normal과 outlier는 시간상 겹치는가?

의도: "둘의 timestamp가 겹치는 파트가 있다"는 가설 A를 판정한다.
겹침에는 세 가지 다른 의미가 있어서 전부 따로 확인한다.

1. **절대 구간 겹침** — `[min,max]` 구간이 교차하는가?
2. **정확 일치** — 동일한 timestamp 값이 실제로 존재하는가?
3. **시각(time-of-day) 겹침** — 날짜를 무시하고 하루 중 시간대가 겹치는가?

(3번을 따로 보는 이유: 플롯의 x축을 시:분:초로만 그리면 날짜가 다른 데이터가 겹쳐 보이는
착시가 흔하다. 겹쳐 보였다면 이 경우일 가능성이 높다.)

In [ ]:
tn, to = DFS["normal"].TimeStamp, DFS["outlier"].TimeStamp

print(f"normal  : {tn.min()}  ~  {tn.max()}")
print(f"outlier : {to.min()}  ~  {to.max()}")
print("-" * 78)
print("1) 절대 구간 겹침       :", max(tn.min(), to.min()) <= min(tn.max(), to.max()))
print("2) 정확히 일치하는 ts 수 :", len(set(tn) & set(to)))
print("   날짜 교집합          :", sorted(set(tn.dt.date) & set(to.dt.date)) or "없음")
a, b = tn.dt.time, to.dt.time
print("3) 시각(time-of-day) 겹침:", max(a.min(), b.min()) <= min(a.max(), b.max()))
print(f"   normal  {a.min()} ~ {a.max()}")
print(f"   outlier {b.min()} ~ {b.max()}")
print("-" * 78)
print("두 파일 사이의 시간 간격 :", to.min() - tn.max())

In [ ]:
# 절대시각 축에 그대로 올려서 확인 (겹침이 있다면 여기서 보여야 한다)
fig, ax = plt.subplots(figsize=(12, 2.0))
for k in DFS:
    t = SEGT[k]
    ax.hlines(np.full(len(t), 0 if k == "normal" else 1),
              t.t_start.values, t.t_end.values, color=COLOR[k], lw=9, label=k)
ax.set_yticks([0, 1]); ax.set_yticklabels(["normal", "outlier"]); ax.set_ylim(-.6, 1.6)
ax.set_title("절대시각 기준 취득 구간 — 두 파일은 5일 떨어져 있고 전혀 겹치지 않는다")
ax.set_xlabel("TimeStamp"); ax.legend(loc="center right")
plt.tight_layout(); plt.show()

### 5-1. 결론 — 가설 A도 기각. 그리고 이게 이 과제의 가장 큰 제약이다

- 절대 구간 겹침 **없음**, 일치하는 timestamp **0개**, 날짜 교집합 **없음**.
- 시각(time-of-day)으로 봐도 겹치지 않는다 (normal 00:00~01:17 심야 / outlier 10:51~10:54 오전).
- 두 파일은 **약 5일 10시간** 떨어져 있다.

**그래서 무엇이 문제인가 — 라벨과 조건이 완전히 교락(confounded)되어 있다**

| | normal | outlier |
|---|---|---|
| 날짜 | 2022-07-12 (화) | 2022-07-17 (일) |
| 시간대 | 00:00 ~ 01:17 (심야) | 10:51 ~ 10:54 (오전) |
| 길이 | 77분 / 599 사이클 | 2.8분 / **21 사이클** |
| 라벨 | 전부 0 | 전부 1 |

"정상"과 "이상"이 **다른 날, 다른 시간대, 다른 길이**로만 관측됐기 때문에,
모델이 잡아낸 차이가 *펌프 고장 때문*인지 *요일·시간대·유온·생산품목·작업자 차이 때문*인지
**이 데이터만으로는 원리적으로 구분할 수 없다.** 이것이 바로 과제가 요구한
"정상적인 운전변화를 이상으로 판단하는 오경보"가 발생하는 메커니즘 그 자체다.

또한 이상 샘플이 **사이클 21개뿐**이라, 사이클 단위 평가에서는 유효 표본이 20개 남짓이다.
정확도/F1을 소수점까지 비교하는 건 의미가 없고, **신뢰구간과 함께 보고**해야 한다.

→ 이 제약은 없앨 수 없으므로, 모델링 단계에서 **정면으로 다룬다**:
   ① 정상 데이터만으로 학습하는 one-class 방식,
   ② 시간대·운전조건에 둔감한 특징 선택,
   ③ normal 내부를 시간으로 분할해 "정상 vs 정상"에서 얼마나 오경보가 나는지를 오경보율의 하한으로 측정.
   ③이 특히 중요하다 — 같은 날 정상 구간끼리도 오경보가 난다면 그 특징은 고장이 아니라 드리프트를 보고 있는 것이다.

## 6. 신호 특성 비교

의도: 이제 세 채널이 실제로 어떻게 다른지 본다. 요약통계 → 분포 → 파형 → 자기상관 순으로
점점 좁혀 들어간다. 자기상관까지 보는 이유는, **평균·분산이 같아도 시간 구조가 다르면
다른 신호**이고, 진동 고장은 보통 진폭보다 *주기성*에서 먼저 드러나기 때문이다.

In [ ]:
desc = pd.concat({k: df[SIGNALS].describe().T for k, df in DFS.items()}, axis=1).round(3)
display(desc)

rows = {}
for k, df in DFS.items():
    col = {}
    for c in SIGNALS:
        x = df[c].values
        col[f"{c} | rms"]    = np.sqrt((x ** 2).mean())
        col[f"{c} | std"]    = x.std(ddof=1)
        col[f"{c} | p99abs"] = np.abs(x).max() if False else np.quantile(np.abs(x), .99)
        col[f"{c} | absmax"] = np.abs(x).max()
    rows[k] = col
stat = pd.DataFrame(rows).round(3)
stat["배율(out/nor)"] = (stat["outlier"] / stat["normal"]).round(2)
display(stat)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.0))
for ax, c in zip(axes, SIGNALS):
    lo = min(DFS[k][c].quantile(.0005) for k in DFS)
    hi = max(DFS[k][c].quantile(.9995) for k in DFS)
    bins = np.linspace(lo, hi, 90)
    for k, df in DFS.items():
        ax.hist(df[c], bins=bins, density=True, alpha=.5, label=k, color=COLOR[k])
    ax.set_yscale("log"); ax.set_title(f"{c} — 값 분포 (y log)")
axes[0].legend()
plt.tight_layout(); plt.show()

의도: 분포만 보면 "outlier가 더 넓게 퍼져 있다"까지만 알 수 있다.
실제 파형을 한 사이클씩 겹쳐 봐야 *어떻게* 다른지가 보인다.
공정한 비교를 위해 양쪽에서 **가장 긴 세그먼트(50샘플=5초)** 를 뽑고, 채널별 y축은 공유한다.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 6), sharex=True)
for j, k in enumerate(DFS):
    sid = SEGT[k].n.idxmax()                      # 가장 긴(=완전한) 세그먼트
    s   = SEG[k][SEG[k].seg_id == sid]
    for i, c in enumerate(SIGNALS):
        axes[i, j].plot(s.t_in_seg, s[c], lw=1.1, color=COLOR[k])
        if j == 0: axes[i, j].set_ylabel(c, fontsize=8)
    axes[0, j].set_title(f"{k} — seg {sid} (n={len(s)})")
    axes[-1, j].set_xlabel("세그먼트 내 경과시간 [s]")

for i in range(3):                                # 채널별 y축 공유 → 스케일 차이를 그대로 드러냄
    lo = min(ax.get_ylim()[0] for ax in axes[i]); hi = max(ax.get_ylim()[1] for ax in axes[i])
    for ax in axes[i]: ax.set_ylim(lo, hi)
plt.tight_layout(); plt.show()

의도: 자기상관(ACF)으로 **시간 구조**를 본다. gap을 가로지르면 안 되므로
세그먼트별로 ACF를 구해 평균한다(길이 40샘플 이상인 세그먼트만 사용).
- 백색잡음이면 lag 1부터 0으로 떨어진다.
- 뚜렷한 주기 성분이 있으면 ACF가 진동한다.

In [ ]:
def acf(x, nlags=30):
    x = np.asarray(x, float) - np.mean(x)
    v = (x * x).sum()
    if v == 0: return np.full(nlags + 1, np.nan)
    return np.array([1.0] + [(x[:-l] * x[l:]).sum() / v for l in range(1, nlags + 1)])

MIN_LEN_ACF = 40
fig, axes = plt.subplots(1, 3, figsize=(13, 3.0))
for ax, c in zip(axes, SIGNALS):
    for k in DFS:
        segs = [g[c].values for _, g in SEG[k].groupby("seg_id") if len(g) >= MIN_LEN_ACF]
        A = np.vstack([acf(s, 30) for s in segs])
        ax.plot(np.arange(31) / FS, np.nanmean(A, axis=0), color=COLOR[k], lw=1.4,
                label=f"{k} (n={len(segs)} seg)")
    ax.axhline(0, color="k", lw=.8)
    ax.set_title(f"세그먼트 평균 ACF — {c}"); ax.set_xlabel("lag [s]"); ax.set_ylim(-1.05, 1.05)
axes[0].legend()
plt.tight_layout(); plt.show()

### 6-1. 확인 결과

**AI0_Vibration (상부 진동) — 가장 해석하기 쉬운 신호**
- RMS 0.071 → 0.448로 약 **6.3배**. 절대 최대값도 0.35 → 1.80으로 5배 이상.
- 더 중요한 건 ACF다. normal은 lag 1에서 이미 거의 0(백색잡음에 가깝다). outlier는 **lag 1에서 0.7 근처**이고
  이후 진동한다 → **뚜렷한 주기 성분이 새로 생겼다.**
  진폭이 아니라 시간 구조가 바뀐 것이므로, 단순 부하 증가나 센서 게인 변화로는 설명되지 않는다.
  → 유압펌프 이상의 물리적 징후로 가장 신뢰할 만한 후보.

**AI1_Vibration (하부 진동) — 보조 신호**
- RMS 0.119 → 0.241로 약 2배. AI0보다 변화폭이 작다. 단독으로는 약하지만 AI0과 조합 시 유용.

**AI2_Current (전류) — 순시값을 그대로 쓰면 안 된다**
- 파형이 부호를 오가며 ±270 범위로 진동하고, ACF도 크고 매끄럽게 진동한다.
  이건 고장 신호가 아니라 **교류 전류를 10 Hz로 샘플링해 생긴 에일리어싱**이다
  (60 Hz를 10 Hz로 샘플링하면 원래 주파수가 복원되지 않고, 아주 작은 주파수 편차가
  느린 가짜 진동으로 나타난다).
- 따라서 **순시값(부호 포함)은 물리적 의미가 없다.** 평균이 normal 1.45 / outlier 57.3으로 달라 보이는 것도
  에일리어싱 위상 차이일 뿐이므로 부하 차이로 해석하면 안 된다.
- 진폭으로 집계해도(사이클 RMS) normal 73~195 vs outlier 64~297로 범위가 크게 겹쳐 분리력이 약하다.
- **단, 파형의 "모양"은 확연히 다르다.** 다음 섹션에서 이 점이 예상 밖의 결과로 이어진다.

## 7. 사이클(세그먼트) 단위 피처와 분리도

의도: 섹션 4의 결론대로 **사이클 = 1 샘플**로 보고 피처를 만든다.
그리고 곧바로 분리도를 재는데, 여기서 보려는 건 "정확도가 얼마나 나오나"가 아니라
**어떤 사이클이 경계에 걸리는가** — 즉 미탐지·오경보가 생길 지점이다.

- 분리도 지표로 **AUC**를 쓴다 (임계값에 의존하지 않아 피처 자체의 힘을 본다).
- 너무 짧은 세그먼트는 통계가 불안정하므로 `n >= 10`(1초 이상)만 사용한다.

In [ ]:
MIN_N = 10   # 1초 미만 세그먼트는 통계가 불안정 → 제외

def seg_features(sdf: pd.DataFrame, label: str) -> pd.DataFrame:
    rows = []
    for sid, g in sdf.groupby("seg_id"):
        r = {"seg_id": sid, "label": label, "n": len(g), "t_start": g.TimeStamp.iloc[0]}
        for c in SIGNALS:
            x, s = g[c].values, g[c]
            r[f"{c}_rms"]    = float(np.sqrt((x ** 2).mean()))
            r[f"{c}_std"]    = float(x.std(ddof=1))
            r[f"{c}_p2p"]    = float(x.max() - x.min())
            r[f"{c}_absmax"] = float(np.abs(x).max())
            r[f"{c}_kurt"]   = float(s.kurt())
            r[f"{c}_ac1"]    = float(s.autocorr(1))
        rows.append(r)
    return pd.DataFrame(rows)

F_all = pd.concat([seg_features(SEG[k], k) for k in DFS], ignore_index=True)
F     = F_all[F_all.n >= MIN_N].reset_index(drop=True)

print(f"전체 세그먼트 {len(F_all)} → n>={MIN_N} 필터 후 {len(F)}")
print("라벨별 세그먼트 수:", F.label.value_counts().to_dict())

FEATS = [c for c in F.columns if c.split("_")[-1] in {"rms", "std", "p2p", "absmax", "kurt", "ac1"}]
display(F.groupby("label")[FEATS].median().round(3).T)

In [ ]:
def auc(x: np.ndarray, y: np.ndarray) -> float:
    # Mann-Whitney U 기반 AUC. y=True가 양성(outlier)
    r = pd.Series(x).rank().values
    n1, n0 = int(y.sum()), int((~y).sum())
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

y = (F.label == "outlier").values
sep = pd.DataFrame({
    "AUC"         : [auc(F[c].values, y) for c in FEATS],
    "normal 중앙값" : [F.loc[~y, c].median() for c in FEATS],
    "outlier 중앙값": [F.loc[y,  c].median() for c in FEATS],
}, index=FEATS)
sep["분리도"] = np.maximum(sep.AUC, 1 - sep.AUC)     # 방향 무관 분리력
display(sep.sort_values("분리도", ascending=False).round(3))

In [ ]:
KEY = "AI0_Vibration_rms"

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, c in zip(axes, [KEY, "AI0_Vibration_ac1", "AI2_Current_rms"]):
    for i, k in enumerate(DFS):
        v = F.loc[F.label == k, c].values
        ax.scatter(np.random.normal(i, .06, len(v)), v, s=14, alpha=.45, color=COLOR[k])
        ax.hlines(np.median(v), i - .25, i + .25, color="k", lw=2)
    ax.set_xticks([0, 1]); ax.set_xticklabels(list(DFS))
    ax.set_title(f"{c}  (AUC={auc(F[c].values, y):.3f})")
axes[0].axhline(F.loc[~y, KEY].max(), color="gray", ls="--", lw=1)
plt.tight_layout(); plt.show()

의도: 여기가 이번 EDA의 결론부다. **정상 사이클의 최댓값**을 가장 보수적인 임계로 잡았을 때
(= 오경보 0을 강제했을 때) **몇 개의 이상 사이클을 놓치는지**를 센다.
이 임계에서도 놓치는 사이클이 있다면, 그건 "임계를 더 낮추면 되는" 문제가 아니라
**정상 분포 안에 완전히 파묻힌 사이클**이라는 뜻이고, 미탐지 원인 분석의 출발점이 된다.

In [ ]:
thr_fp0 = F.loc[~y, KEY].max()          # 오경보 0을 강제하는 임계
miss    = F[(F.label == "outlier") & (F[KEY] <= thr_fp0)]

print(f"임계: {KEY} > {thr_fp0:.4f}  (= 정상 사이클 최댓값, 오경보 0 강제)")
print(f"  이상 사이클 탐지 : {int(y.sum()) - len(miss)} / {int(y.sum())}")
print(f"  미탐지           : {len(miss)} / {int(y.sum())}")
print()
print("정상 사이클 상위 5개 (오경보에 가장 가까운 사이클):")
display(F[~y].nlargest(5, KEY)[["seg_id", "n", "t_start", KEY, "AI0_Vibration_ac1"]])
print("미탐지된 이상 사이클:")
display(miss[["seg_id", "n", "t_start", KEY, "AI0_Vibration_ac1", "AI2_Current_rms"]])

In [ ]:
# 미탐지 사이클의 파형을 직접 본다 — 정말 정상처럼 보이는가?
if len(miss):
    ids = miss.seg_id.tolist()[:3]
    fig, axes = plt.subplots(1, len(ids) + 1, figsize=(3.4 * (len(ids) + 1), 2.6), sharey=True)
    ref = SEG["normal"][SEG["normal"].seg_id == SEGT["normal"].n.idxmax()]
    axes[0].plot(ref.t_in_seg, ref.AI0_Vibration, lw=1, color=COLOR["normal"])
    axes[0].set_title("참고: 정상 사이클"); axes[0].set_ylabel("AI0_Vibration")
    for ax, sid in zip(axes[1:], ids):
        s = SEG["outlier"][SEG["outlier"].seg_id == sid]
        ax.plot(s.t_in_seg, s.AI0_Vibration, lw=1, color=COLOR["outlier"])
        ax.set_title(f"미탐지 outlier seg {sid} (n={len(s)})")
    for ax in axes: ax.set_xlabel("t [s]")
    plt.tight_layout(); plt.show()
else:
    print("미탐지 없음")

의도: AUC는 임계값과 무관한 지표라 **실제 운영 지점**을 말해주지 않는다.
"오경보 0"을 요구했을 때 피처별로 몇 개를 탐지하는지 직접 비교한다.
AUC 순위와 이 순위가 다르면, 피처 선택 기준을 AUC로 잡으면 안 된다는 뜻이다.

In [ ]:
def tpr_at_zero_fp(col, direction=">"):
    if direction == ">":
        thr = F.loc[~y, col].max(); det = F.loc[y, col] > thr
    else:
        thr = F.loc[~y, col].min(); det = F.loc[y, col] < thr
    return pd.Series({"방향": f"{direction} {thr:.4f}",
                      "탐지": f"{int(det.sum())} / {int(y.sum())}",
                      "탐지율": round(det.mean(), 3),
                      "미탐지 seg": ", ".join(map(str, F.loc[y][~det].seg_id.tolist()))})

ops = pd.DataFrame({
    "AI0_Vibration_rms"  : tpr_at_zero_fp("AI0_Vibration_rms",  ">"),
    "AI0_Vibration_ac1"  : tpr_at_zero_fp("AI0_Vibration_ac1",  ">"),
    "AI1_Vibration_rms"  : tpr_at_zero_fp("AI1_Vibration_rms",  ">"),
    "AI2_Current_ac1"    : tpr_at_zero_fp("AI2_Current_ac1",    "<"),
}).T
_a = [auc(F[c].values, y) for c in ops.index]
ops.insert(0, "분리도", [round(max(v, 1 - v), 3) for v in _a])   # 방향 보정
ops.insert(0, "AUC", [round(v, 3) for v in _a])                  # 원값(0에 가까우면 역방향)
print("오경보 0을 강제했을 때의 탐지 성능 (임계 = 정상 사이클의 극값)")
display(ops)

의도: 섹션 5에서 지적한 교락을 **정상 데이터만으로** 미리 진단한다.
normal 599 사이클을 시간순 4등분해 구간별 피처 중앙값을 본다.
정상인데도 구간마다 값이 흔들린다면, 그 피처는 고장이 아니라 **운전조건 드리프트**를 보고 있는 것이고
→ 그대로 쓰면 오경보의 직접 원인이 된다.

In [ ]:
q = pd.qcut(F.loc[~y, "t_start"].rank(method="first"), 4, labels=["Q1", "Q2", "Q3", "Q4"])
drift = F[~y].groupby(q.values)[["AI0_Vibration_rms", "AI0_Vibration_ac1",
                                 "AI1_Vibration_rms", "AI2_Current_rms",
                                 "AI2_Current_ac1"]].median().T
drift["Q4/Q1"] = (drift["Q4"] / drift["Q1"]).round(2)
print("normal 파일 내부 시간 4분할 — 구간별 중앙값 (정상인데도 변하는가?)")
display(drift.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 2.8))
for ax, c in zip(axes, ["AI0_Vibration_rms", "AI2_Current_rms"]):
    ax.plot(F.loc[~y, "t_start"], F.loc[~y, c], ".", ms=4, alpha=.5, color=COLOR["normal"])
    ax.set_title(f"normal 내부 시간 추이 — {c}"); ax.set_xlabel("TimeStamp")
plt.tight_layout(); plt.show()

### 7-1. 확인 결과 — 예상과 달랐던 부분

**(1) 가장 잘 분리하는 피처는 진폭이 아니라 "파형의 모양"이다**

| 피처 | 분리도 | normal 중앙값 | outlier 중앙값 |
|---|---|---|---|
| `AI2_Current_ac1` | **1.000** | 0.928 | 0.439 |
| `AI2_Current_kurt` | 0.979 | -1.482 | -0.292 |
| `AI0_Vibration_ac1` | 0.962 | -0.032 | 0.694 |
| `AI1_Vibration_ac1` | 0.922 | 0.256 | 0.669 |
| `AI0_Vibration_rms` | 0.849 | 0.070 | 0.356 |
| `AI2_Current_rms` | 0.635 | 100.4 | 149.5 |

RMS가 6.3배 차이 나는 `AI0_Vibration_rms`의 AUC가 0.849에 그친 반면,
**자기상관·첨도 같은 형태 피처가 위를 모두 차지했다.**
(분리도는 방향 보정값 `max(AUC, 1-AUC)`이다. `AI2_Current_ac1`은 원 AUC가 0.000 —
즉 normal이 outlier보다 *높은* 쪽으로 완전히 역방향 분리된다.) 진폭은 정상 사이클끼리도 편차가 커서
겹치는 반면, 주기성은 정상에서 일관되게 낮기 때문이다.

**(2) `AI2_Current_ac1`의 AUC 1.000은 성과가 아니라 경고 신호로 읽어야 한다**

정상 사이클의 전류 ac1은 [0.757, 0.987]에 매우 좁게 몰려 있고, 77분 내내 거의 변하지 않는다
(시간 4분할 중앙값 0.928 / 0.929 / 0.929 / 0.926). 반면 outlier는 중앙값 0.439로 크게 낮다.
그런데 **6-1에서 본 대로 이 채널은 에일리어싱된 신호**다. 에일리어싱 주파수는
실제 전원 주파수와 DAQ 클럭의 미세한 편차로 정해지므로, **고장이 없어도 날짜가 바뀌면 달라질 수 있다.**
AUC가 거의 1.0이라는 건 오히려 "라벨이 날짜와 교락돼 있다"(섹션 5)는 신호일 수 있다.

다만 반대 증거도 있다. 주파수만 이동했다면 파형은 여전히 깨끗한 정현파이므로 첨도가
정현파 이론값(-1.5) 근처에 머물러야 하는데, 실제로는 -1.482 → -0.292로 올라갔다.
즉 **전류에 광대역 잡음 성분이 실제로 추가됐다**는 뜻이고, 이는 펌프 맥동·캐비테이션의 징후로 볼 여지가 있다.
→ 결론을 내리기에는 근거가 부족하다. **02에서 전류 채널은 별도로 검증하기 전까지 주 피처로 채택하지 않는다.**

**(3) AUC 순위와 실제 운영 지점(오경보 0)의 순위가 다르다**

오경보를 0으로 강제하는 임계(정상 사이클 최댓값)에서의 탐지 성능은 순위가 뒤집힌다.

| 임계 | 탐지 | 미탐지 사이클 |
|---|---|---|
| `AI0_Vibration_rms` > 0.1478 | **14 / 17** | seg 3, 19, 20 |
| `AI2_Current_ac1` < 0.7566 | 15 / 17 | seg 15, 17 |
| `AI0_Vibration_ac1` > 0.6982 | **8 / 17** | 9개 |

`AI0_Vibration_ac1`은 AUC가 0.962로 높지만, 정상 중에도 ac1이 0.698까지 올라가는 사이클이 있어
**오경보 0을 요구하면 절반 이상을 놓친다.** 두 피처를 OR로 묶어도 14/17로 개선이 없다.
→ **AUC로 피처를 고르고 끝내면 안 된다.** 이 과제의 평가 기준은 "고정 오경보율에서의 탐지율"이어야 한다.

**(4) 이상은 상시가 아니라 간헐적이다**

가장 보수적인 임계에서도 17개 중 3개(seg 3, 19, 20)가 미탐지로 남고, 파형을 보면 진폭·주기성 모두
정상과 구분되지 않는다. **단일 사이클 판정으로는 원리적으로 놓칠 수밖에 없다.**
→ "최근 N사이클 중 M회 초과 시 경보" 같은 누적 판정이 필요하다. 이는 미탐지와 오경보를 동시에 줄이는 방향이다.

## 8. 정리 및 다음 단계

### 확인된 사실

| # | 항목 | 결과 |
|---|---|---|
| 1 | 샘플링 | 0.1초(10 Hz) 고정(97%), null 0, 시간 오름차순 정상 |
| 2 | 중복 | normal에 값까지 동일한 중복 1쌍 (idx 11564/11565) → 제거 대상 |
| 3 | 라벨 | `Equipment_state`는 **파일 단위 상수** → 행 단위 정답 없음, one-class 접근 필요 |
| 4 | gap | **결측이 아니라 취득 duty cycle**. 세그먼트 길이 상한 50샘플, 사이클 주기 약 8초, duty 36~43% |
| 5 | 가설 A (timestamp 겹침) | **기각** — 5일 9시간 34분 떨어져 있고, 시각대(심야 vs 오전)도 겹치지 않음 |
| 6 | 가설 B (outlier의 빈 구간 = 삭제된 normal) | **기각** — normal에도 동일 구조의 gap 598개 존재 |
| 7 | 최강 피처 | 진폭이 아니라 **형태 피처**(`AI2_Current_ac1` AUC 1.000, `AI0_Vibration_ac1` 0.962) |
| 8 | 전류 | 에일리어싱된 교류. 순시값 사용 금지. ac1의 완벽한 분리는 **교락 의심 신호**로 취급 |
| 9 | 운영 지점 | 오경보 0에서는 `AI0_Vibration_rms`가 14/17로 최선. **AUC 순위와 불일치** |
| 10 | 미탐지 | 가장 보수적 임계에서도 3개(seg 3, 19, 20)는 정상과 구분 불가 → **이상은 간헐적** |
| 11 | 정상 내부 드리프트 | normal 77분 안에서도 `AI0_Vibration_rms`가 Q3→Q4에 0.076→0.045로 떨어짐 → 국면이 섞여 있음 |

### 3번 질문에 대한 답

두 파일은 **시간상 전혀 겹치지 않고**, outlier의 빈 구간도 **normal이 삭제된 것이 아니다.**
양쪽 모두 같은 취득 설정(약 8초 주기, 최대 50샘플 버스트)에서 나온 **서로 다른 날의 독립된 기록**이다.

겹쳐 보였던 원인으로 가능성이 높은 것: ① x축을 시:분:초로만 그렸거나 ② 행 인덱스(0..N)를 x축으로 써서
두 파일이 같은 구간에 그려진 경우. 섹션 5의 절대시각 플롯이 이 둘을 구분해 준다.

그런데 **"겹치지 않는다"는 사실 자체가 이 과제의 핵심 난점이다.**
정상/이상이 날짜·시간대와 완전히 교락되어 있어 모델이 잡은 차이가 고장 때문인지
운전조건 차이 때문인지 이 데이터만으로는 분리할 수 없다.
`AI2_Current_ac1`의 AUC 1.000이 바로 그 위험을 보여주는 사례다 —
과제가 요구한 "정상적인 운전변화를 이상으로 오인하는 오경보"는 여기서 발생한다.

### 4번 — 다음에 확인할 것 (우선순위 순)

1. **정상 내부 오경보율 측정 (최우선).** normal 599 사이클을 시간순 앞 70% / 뒤 30%로 나눠,
   앞으로 기준을 학습하고 뒤를 채점한다. **정상 vs 정상인데 경보가 나면** 그 피처는 고장이 아니라
   드리프트를 보고 있는 것이다. 섹션 7의 드리프트 점검에서 Q4 구간의 `AI0_Vibration_rms`가
   이미 40% 떨어지는 게 확인됐으므로, 이 검증은 형식이 아니라 실제로 걸릴 가능성이 높다.
2. **정상 데이터 내 운전 국면 분할.** 위 드리프트가 웜업·공회전·부하변동 중 무엇인지 규명한다.
   국면이 섞여 있다면 단일 기준선이 아니라 **국면별 기준선**(또는 국면 정규화)이 필요하다.
3. **전류 채널 검증 또는 배제.** `AI2_Current_ac1`이 고장 때문인지 에일리어싱 조건 변화 때문인지
   가른다. 정상 파일 내부를 앞/뒤로 나눠 ac1이 얼마나 흔들리는지 재고(현재 0.928→0.926으로 매우 안정),
   첨도 상승(-1.48 → -0.29)이 유지되는지 확인한다. 판단이 서지 않으면 **주 피처에서 제외**한다.
4. **세그먼트 길이 편향 점검.** 세그먼트 길이가 10~50샘플로 들쭉날쭉하다. 길이가 짧을수록 RMS·ac1 추정이
   불안정해지므로, 피처가 *길이*를 보고 있지 않은지 확인하고 필요하면 고정 길이(예: 앞 30샘플)로 잘라 쓴다.
   미탐지 사이클 중 2개(seg 19: n=10, seg 20: n=15)가 짧은 세그먼트라는 점이 특히 의심스럽다.
5. **주파수 영역 분석.** 10 Hz 샘플링이라 나이퀴스트가 5 Hz로 낮지만, AI0의 ac1 상승이 시사하는
   0.5~2 Hz 성분은 볼 수 있다. 세그먼트별 PSD로 **이상 시 어느 대역이 올라오는지** 특정한다.
   물리적 해석(맥동/캐비테이션/베어링)이 붙으면 오경보 판별 근거가 훨씬 강해진다.
6. **누적 판정 로직 설계.** 10번 결과대로 이상은 간헐적이다. 사이클 점수에 "최근 N사이클 중 M회 초과"
   규칙을 씌워 미탐지와 오경보가 동시에 최소가 되는 (N, M)을 찾는다.
7. **평가 프로토콜 확정.** 이상 사이클이 17개뿐이라 점추정이 불안정하다. 부트스트랩 신뢰구간을 붙이고,
   지표는 정확도·F1이 아니라 **고정 오경보율에서의 탐지율**로 보고한다.
8. **(확보 가능하다면) 같은 날·같은 시간대의 정상 구간.** 교락을 깨는 유일한 근본 해법이다.
   2022-07-17 오전의 정상 데이터가 있다면 최우선으로 요청할 가치가 있다.